# 99 - Aggregate Results (run after all model notebooks)

Runs on **CPU** (Accelerator = None). On Kaggle: **Add Input > Datasets** and add `aggregate-dataset` (path `/kaggle/input/datasets/falaqueabrar69/aggregate-dataset`, the folder with the 4 models' result files). Optionally also add `chittagonian-splits` to double-check the test-set fingerprint. It reads only files produced by notebooks 01-04 from their `results_v5/` folders and **never generates or edits a score**. Everything comes from the saved predictions.

Outputs: `all_results_v5.csv`, `results_table_v5.tex`, `significance_v5.csv`, 4 metric charts and one combined loss chart (300 DPI).


**NFC re-scoring version (2026-11-22).** Cell 2b re-scores BLEU, chrF, ROUGE-L and BERTScore from the saved `preds_*.csv` files after Unicode NFC normalization (references mix one-code and two-code forms of য়/ড়; models always output the two-code form). No model is retrained. **Turn Internet ON** (Settings > Internet) so pip and the BERTScore model (`xlm-roberta-base`) can download. CPU is enough; a GPU makes BERTScore faster. Old outputs are copied to `results_v5/raw_unnormalized_backup/` first.


In [ ]:
# -- CELL 1: Environment -------------------------------------------------
# FIX (NFC re-score): evaluate / rouge_score / bert_score are needed by Cell 2b.
# transformers + torch are already installed on Kaggle, so they are not reinstalled.
!pip install -q sacrebleu pandas matplotlib seaborn jinja2 evaluate rouge_score bert_score


In [ ]:
# -- CELL 2: Load Per-Model Results --------------------------------------
import os, glob, json, shutil, numpy as np, pandas as pd
project_dir = '/kaggle/working/Chittagonian_AI_Project'
results_dir = f'{project_dir}/results_v5'
os.makedirs(results_dir, exist_ok=True)

# Your dataset with the result files of the 4 models (made from results_v5.zip)
RESULTS_DATASET = '/kaggle/input/datasets/falaqueabrar69/aggregate-dataset'

def _has_scores(d):
    return bool(glob.glob(f'{d}/baseline_scores_*.csv') or glob.glob(f'{d}/finetuned_scores_*.csv'))

_srcs = [d for d in (f'{RESULTS_DATASET}/results_v5', RESULTS_DATASET) if os.path.isdir(d) and _has_scores(d)]
if not _srcs:   # dataset has another folder layout: search for the score files anywhere inside it
    _srcs = sorted({os.path.dirname(p) for pat in ('baseline_scores_*.csv', 'finetuned_scores_*.csv')
                    for p in glob.glob(f'{RESULTS_DATASET}/**/{pat}', recursive=True)})
if not _srcs:   # last resort: any results_v5 folder under /kaggle/input (e.g. outputs of notebooks 01-04)
    _srcs = sorted(d for d in glob.glob('/kaggle/input/**/results_v5', recursive=True) if _has_scores(d))
assert _srcs, (f"No result files found in {RESULTS_DATASET}. Add the dataset 'aggregate-dataset' to this notebook "
               "(Add Input > Datasets); it must contain baseline_scores_*.csv, finetuned_scores_*.csv, preds_*.csv, loss_hist_*.json.")
for _s in _srcs:
    for _f in glob.glob(f'{_s}/*'):
        if os.path.isfile(_f): shutil.copy2(_f, results_dir)
print("Collected results from:"); [print("  -", s) for s in _srcs]

MODELS = ["mT5-Small", "BanglaT5", "NLLB-200", "Bangla-BERT2BERT"]   # paper order
COLORS = {"mT5-Small": "#2980b9", "BanglaT5": "#8e44ad",
          "NLLB-200": "#16a085", "Bangla-BERT2BERT": "#e67e22"}
METRICS = ["BLEU", "ROUGE-L", "chrF", "BERTScore (F1)"]

files = sorted(glob.glob(f'{results_dir}/baseline_scores_*.csv') +
               glob.glob(f'{results_dir}/finetuned_scores_*.csv'))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
missing = [(m, s) for m in MODELS for s in ["Zero-Shot", "Fine-Tuned"]
           if df[(df.Model == m) & (df.State == s)].empty]
if missing:
    print("[WARN] Not run yet (left blank, never filled in):", missing)

# Every model must have been scored on the same test set
SPLITS_DATASET = '/kaggle/input/datasets/falaqueabrar69/chittagonian-splits'
_mp = [p for p in (f'{SPLITS_DATASET}/Chittagonian_AI_Project/data_manifest.json', f'{SPLITS_DATASET}/data_manifest.json')
       if os.path.exists(p)] or sorted(glob.glob('/kaggle/input/**/data_manifest.json', recursive=True))
if _mp:
    manifest = json.load(open(_mp[0], encoding='utf-8'))
else:   # chittagonian-splits not attached
    _rc = sorted(glob.glob(f'{results_dir}/run_config_*.json'))
    if _rc:
        manifest = {"test": json.load(open(_rc[0], encoding='utf-8'))["data"]["test"]}
        print("[INFO] data_manifest.json not found; using the test fingerprint from the first run record.")
    else:
        manifest = {"test": {"rows": int(df["n_test"].dropna().iloc[0]), "md5": None}}
        print("[WARN] No data_manifest.json and no run_config_*.json: test-set fingerprint cannot be checked; "
              "using the test size stored in the score files.")
_rcs = sorted(glob.glob(f'{results_dir}/run_config_*.json'))
for f in _rcs:
    rc = json.load(open(f))
    assert rc["data"]["test"]["md5"] == manifest["test"]["md5"], f"{f}: different test set!"
    assert rc["decoding"]["num_beams"] == 4, f"{f}: different decoding!"
if _rcs:
    print(f"[OK] {len(_rcs)} run records use the same test set and decoding.")
else:
    print("[WARN] No run_config_*.json files: decoding and test-set fingerprint were not verified from run records.")
_n = df["n_test"].dropna().unique() if "n_test" in df else []
assert len(_n) <= 1, f"Models were scored on different test sizes: {list(_n)}"
if len(_n) == 1: print(f"[OK] Every score file reports the same test size: {int(_n[0])} sentences.")
if (df.get("n_failed", 0) > 0).any():
    print("[WARN] Some predictions failed (empty) - see n_failed column:"); display(df[df.n_failed > 0])


In [ ]:
# -- CELL 2b: Re-score ALL metrics with Unicode NFC (no retraining) -------
# Why: 435/1,162 references store য়/ড় as ONE code (U+09DF/U+09DC); every model
# outputs the SAME letters as TWO codes (য/ড + nukta U+09BC). They look identical
# and Unicode says they are equivalent, but BLEU/chrF/ROUGE compare raw codes and
# count correct words as wrong. NFC makes both sides use the same codes.
# Metric settings are copied unchanged from Cell 3 of the model notebooks (01-04).
import unicodedata, torch, evaluate
from sacrebleu.metrics import BLEU, CHRF

NFC = lambda s: unicodedata.normalize("NFC", str(s)).strip()

# 1) Back up the old (un-normalized) outputs before Cells 3-5 overwrite them
bk_dir = f'{results_dir}/raw_unnormalized_backup'
os.makedirs(bk_dir, exist_ok=True)
for f in ["all_results_v5.csv", "significance_v5.csv", "results_table_v5.tex"] + \
         [os.path.basename(p) for p in glob.glob(f'{results_dir}/chart_*.png')] + \
         [os.path.basename(p) for p in glob.glob(f'{results_dir}/*_scores_*.csv')]:
    if os.path.exists(f'{results_dir}/{f}') and not os.path.exists(f'{bk_dir}/{f}'):
        shutil.copy(f'{results_dir}/{f}', f'{bk_dir}/{f}')
print(f"[OK] Old outputs backed up in {bk_dir}")

# 2) Same metric settings as the model notebooks
_bleu, _chrf = BLEU(), CHRF()
rouge_m = evaluate.load("rouge")
bert_m  = evaluate.load("bertscore")
bs_device = "cuda" if torch.cuda.is_available() else "cpu"

def score_nfc(preds, refs):
    preds, refs = [NFC(p) for p in preds], [NFC(r) for r in refs]      # FIX: NFC both sides
    bleu = _bleu.corpus_score(preds, [refs]).score
    chrf = _chrf.corpus_score(preds, [refs]).score
    rl   = rouge_m.compute(predictions=preds, references=refs,
                           tokenizer=lambda x: x.split())['rougeL']    # word-level, Bengali-safe
    try:
        bs   = bert_m.compute(predictions=preds, references=refs, lang="bn",
                              model_type="xlm-roberta-base", batch_size=32, device=bs_device)
        bert = round(float(np.mean(bs['f1'])) * 100, 2)
    except Exception as e:
        print(f"  [WARN] BERTScore: {e}"); bert = float('nan')
    return {"BLEU": round(bleu, 2), "ROUGE-L": round(rl * 100, 2),
            "chrF": round(chrf, 2), "BERTScore (F1)": bert}

# 3) Re-score every saved prediction file and replace the scores in `df`
nfc_dir = f'{results_dir}/nfc_rescored'      # separate folder -> Cell 2's glob never mixes old + new
os.makedirs(nfc_dir, exist_ok=True)
old_df, new_rows = df.copy(), []
for m in MODELS:
    for s, tag, prefix in [("Zero-Shot", "zeroshot", "baseline"), ("Fine-Tuned", "finetuned", "finetuned")]:
        p = f'{results_dir}/preds_{tag}_{m}.csv'
        if not os.path.exists(p): continue
        d = pd.read_csv(p, keep_default_na=False)
        print(f"-- Re-scoring {m} / {s} ({len(d)} sentences, BERTScore on {bs_device})")
        row = {"Model": m, "State": s,
               **score_nfc(d['prediction'].astype(str).tolist(), d['reference'].astype(str).tolist())}
        old = old_df[(old_df.Model == m) & (old_df.State == s)]
        row["n_test"]   = len(d)
        row["n_failed"] = int(pd.to_numeric(old.n_failed, errors="coerce").fillna(0).values[0]) \
                          if len(old) and "n_failed" in old else 0
        pd.DataFrame([row]).to_csv(f'{nfc_dir}/{prefix}_scores_{m}.csv', index=False)
        new_rows.append(row)
assert new_rows, f"No preds_*.csv files found in {results_dir}"
df = pd.DataFrame(new_rows)      # Cells 3-6 now use the NFC scores
assert df[df.State == "Fine-Tuned"]["BERTScore (F1)"].notna().any(), \
    "BERTScore failed for every model (see [WARN] lines). On Kaggle: Settings > Internet ON, then re-run."

# 4) Old vs new side by side (old = un-normalized). Expected fine-tuned BLEU (NFC):
#    BanglaT5 61.54, NLLB-200 60.93, mT5-Small 51.51, Bangla-BERT2BERT 35.17
cmp = old_df.merge(df, on=["Model", "State"], suffixes=(" old", " NFC"))
show = ["Model", "State"] + [f"{c} {t}" for c in METRICS for t in ["old", "NFC"]]
pd.set_option("display.max_columns", None); pd.set_option("display.width", 250)
display(cmp[show])
cmp[show].to_csv(f'{nfc_dir}/old_vs_nfc_scores.csv', index=False)
print("[OK] Scores replaced with NFC versions. Cells 3-7 now build CIs, tables and charts from them.")


In [ ]:
# -- CELL 3: 95% Confidence Intervals + Paired Significance (bootstrap) ---
# Why: a journal reviewer will ask "is a 1-BLEU gap real or luck?". We resample the
# test sentences 1,000 times (Koehn, 2004, paired bootstrap resampling) and recompute the
# corpus score each time. Uses sacreBLEU's own sentence statistics -> exact corpus BLEU/chrF.
from sacrebleu.metrics import BLEU, CHRF
N_BOOT, rng_seed = 1000, 42

def load_preds(model, state):
    tag = "zeroshot" if state == "Zero-Shot" else "finetuned"
    p = f'{results_dir}/preds_{tag}_{model}.csv'
    if not os.path.exists(p): return None
    d = pd.read_csv(p, keep_default_na=False)
    return [NFC(x) for x in d['prediction']], [NFC(x) for x in d['reference']]   # FIX: NFC both sides (same as Cell 2b)

def sent_stats(metric, hyps, refs):
    return np.array(metric._extract_corpus_statistics(hyps, [refs]))

def corpus_from_stats(metric, stats):
    return metric._compute_score_from_stats(list(stats.sum(axis=0))).score

metrics = {"BLEU": BLEU(), "chrF": CHRF()}
rng   = np.random.default_rng(rng_seed)
n_test = manifest["test"]["rows"]
IDX   = rng.integers(0, n_test, size=(N_BOOT, n_test))      # same resamples for every system -> paired

stats, boot = {}, {}
for m in MODELS:
    for s in ["Zero-Shot", "Fine-Tuned"]:
        pr = load_preds(m, s)
        if pr is None: continue
        for name, met in metrics.items():
            st = sent_stats(met, *pr)
            stats[(m, s, name)] = st
            boot[(m, s, name)] = np.array([corpus_from_stats(met, st[i]) for i in IDX])

def ci(key):
    b = boot.get(key)
    return (np.nan, np.nan) if b is None else tuple(np.percentile(b, [2.5, 97.5]).round(2))

for name in metrics:
    df[f"{name} 95% CI"] = [f"[{ci((r.Model, r.State, name))[0]}, {ci((r.Model, r.State, name))[1]}]"
                            for r in df.itertuples()]

# Paired test: best fine-tuned model vs every other fine-tuned model
ft = df[df.State == "Fine-Tuned"]
rows = []
if not ft.empty:
    best = ft.sort_values("BLEU", ascending=False).iloc[0].Model
    for other in ft.Model:
        if other == best: continue
        for name in metrics:
            a, b = boot.get((best, "Fine-Tuned", name)), boot.get((other, "Fine-Tuned", name))
            if a is None or b is None: continue
            p = (np.sum(b >= a) + 1) / (N_BOOT + 1)           # one-sided, add-one smoothing
            rows.append({"Best": best, "Compared": other, "Metric": name,
                         "Mean diff": round(float(np.mean(a - b)), 2), "p-value": round(float(p), 4),
                         "Significant (p<0.05)": p < 0.05})
sig_df = pd.DataFrame(rows)
sig_df.to_csv(f'{results_dir}/significance_v5.csv', index=False)
display(sig_df)


In [ ]:
# -- CELL 4: Master Table (CSV + LaTeX) ----------------------------------
df["Model"] = pd.Categorical(df["Model"], MODELS, ordered=True)
df["State"] = pd.Categorical(df["State"], ["Zero-Shot", "Fine-Tuned"], ordered=True)
df = df.sort_values(["Model", "State"]).reset_index(drop=True)

# best-epoch / training time from run records (blank if a model was not trained)
extra = {}
for m in MODELS:
    p = f'{results_dir}/loss_hist_{m}.json'
    if os.path.exists(p):
        h = json.load(open(p)); extra[m] = (h["best_epoch"], h["epochs_run"], h["train_minutes"])
df["Best epoch"]   = [extra.get(m, (None,))[0] if s == "Fine-Tuned" else None for m, s in zip(df.Model, df.State)]
df["Train (min)"]  = [extra.get(m, (None, None, None))[2] if s == "Fine-Tuned" else None for m, s in zip(df.Model, df.State)]

pd.set_option('display.float_format', '{:.2f}'.format)
cols = ["Model", "State"] + METRICS + ["BLEU 95% CI", "chrF 95% CI", "Best epoch", "Train (min)"]
print("=" * 90); print("  ALL MODELS - TEST SET (identical split + decoding)"); print("=" * 90)
display(df[cols])
df[cols].to_csv(f'{results_dir}/all_results_v5.csv', index=False)

latex = df[["Model", "State"] + METRICS].to_latex(index=False, float_format="%.2f",
          caption="Chittagonian -> Standard Bangla test-set results (beam=4, max length 64).",
          label="tab:results")
open(f'{results_dir}/results_table_v5.tex', 'w', encoding='utf-8').write(latex)
print(f"[OK] Saved all_results_v5.csv and results_table_v5.tex in {results_dir}")


In [ ]:
# -- CELL 5: Grouped Bar Charts (Zero-Shot vs Fine-Tuned, with 95% CI) ---
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})
TITLES = {"BLEU": "Translation Accuracy: BLEU", "ROUGE-L": "Word-Level Overlap: ROUGE-L",
          "chrF": "Character-Level Score: chrF", "BERTScore (F1)": "Semantic Similarity: BERTScore F1 (x100)"}
x, width = np.arange(len(MODELS)), 0.36

def val(m, s, col):
    r = df[(df.Model == m) & (df.State == s)]
    return float(r[col].values[0]) if len(r) and pd.notna(r[col].values[0]) else np.nan

for col in METRICS:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for k, (s, color) in enumerate([("Zero-Shot", "#bdc3c7"), ("Fine-Tuned", "#27ae60")]):
        v = [val(m, s, col) for m in MODELS]
        err = None
        if col in metrics:     # CI error bars only where we bootstrapped
            lo = [v[i] - ci((m, s, col))[0] for i, m in enumerate(MODELS)]
            hi = [ci((m, s, col))[1] - v[i] for i, m in enumerate(MODELS)]
            err = np.clip(np.nan_to_num(np.array([lo, hi])), 0, None)   # error bars must not be negative (bootstrap CI can sit slightly off a tiny score)
        bars = ax.bar(x + (k - 0.5) * width, np.nan_to_num(v), width, yerr=err, capsize=3,
                      label=s, color=color, edgecolor="white", linewidth=2)
        ax.bar_label(bars, labels=["n/a" if np.isnan(t) else f"{t:.2f}" for t in v],
                     padding=3, fontsize=9)
    ax.set_title(TITLES[col], fontweight="bold", pad=12); ax.set_ylabel(col)
    ax.set_xticks(x); ax.set_xticklabels(MODELS, fontweight="bold")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left", frameon=False); sns.despine(); plt.tight_layout()
    sc = col.replace(' ', '_').replace('(', '').replace(')', '')
    path = f'{results_dir}/chart_{sc}.png'
    plt.savefig(path, dpi=300, bbox_inches="tight"); plt.show(); plt.close()
    print(f"Saved: {path}")


In [ ]:
# -- CELL 6: Validation Loss of All Models in One Figure -----------------
fig, ax = plt.subplots(figsize=(10, 5.5))
for m in MODELS:
    p = f'{results_dir}/loss_hist_{m}.json'
    if not os.path.exists(p): continue
    h = json.load(open(p))
    ep = range(1, len(h["eval_loss"]) + 1)
    ax.plot(ep, h["eval_loss"], marker="o", lw=2, color=COLORS[m], label=f"{m} (best ep {h['best_epoch']})")
    ax.plot(h["best_epoch"], h["best_eval_loss"], marker="*", ms=14, color=COLORS[m])
ax.set_title("Validation Loss per Epoch (* = checkpoint used for testing)", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation loss"); ax.legend(frameon=False); sns.despine()
plt.tight_layout()
path = f'{results_dir}/val_loss_all_models.png'
plt.savefig(path, dpi=300, bbox_inches="tight"); plt.show(); plt.close()
print(f"Saved: {path}")
print("Note: losses of different tokenizers are NOT directly comparable across models; "
      "use this figure only to show convergence, not to rank models.")


In [ ]:
# -- CELL 7: Package everything for download -------------------------------------
import zipfile
zpath = '/kaggle/working/results_v5_all.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(results_dir):          # FIX: also zip the sub-folders nfc_rescored/ and raw_unnormalized_backup/
        for f in sorted(fs):
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, results_dir))
print(f"[OK] {zpath}  ({os.path.getsize(zpath)/1024:.0f} KB)")